# Respuestas: Tarea — Sistemas de Recomendación

Este notebook recoge las respuestas de la tarea de sistemas de recomendación y usa los datos almacenados en `data/raw/`: `movies.csv`, `ratings.csv`, `tags.csv` y `links.csv`.

## Actividad 1 — Exploración inicial

- `movies_dataset`: contiene, para cada película, su identificador `movieId`, su `title` (a veces con año entre paréntesis) y la columna `genres` con los géneros separados por `|`.
- `ratings_dataset`: contiene las calificaciones explícitas: `userId`, `movieId`, `rating` (float) y `timestamp`.
- `tags_dataset`: contiene etiquetas libres asignadas por usuarios: `userId`, `movieId`, `tag` y `timestamp`.
- Variables que permiten relacionar los archivos: `movieId` (une `movies` con `ratings` y `tags`) y `userId` (une `ratings` y `tags` por usuario).

Contadores (calculados sobre los CSV de este repositorio):

- `num_users` = 610
- `num_movies` = 9742
- `num_ratings` = 100836
- `num_tags` = 3683

- Archivo más importante para filtro colaborativo: `ratings.csv`, porque contiene las calificaciones que permiten medir similitud entre usuarios.
- Archivo más importante para filtro basado en contenido: `movies.csv` (la columna `genres`), aunque `tags.csv` añade información de texto que puede enriquecer el perfil de la película.

## Actividad 2 — Matriz usuario–película

- `user_dataset` es la tabla pivote con índices `userId`, columnas `movieId` y valores la `rating`.
- Cada fila representa el perfil de un usuario: las coordenadas son las calificaciones que ese usuario dio a cada película.
- Cada columna representa una película (`movieId`).
- Los valores iguales a 0 indican ausencia de calificación (usuario no vio o no evaluó la película). En esta práctica 0 se usa como marcador numérico para 
.
- Diferencia entre película no vista y película mal calificada: `no vista` → valor 0 (ausencia de dato). `mal calificada` → valor numérico bajo (por ejemplo 1.0) — semánticamente son distintos y por eso en la comparación debemos tratar los ceros como 
 y no como una calificación negativa.

## Actividad 3 — Interpretación del gráfico (toy_group)

- Cada punto corresponde a un usuario cuya coordenada X es la calificación para `movieId = 1` y la coordenada Y la calificación para `movieId = 2`.
- Que dos usuarios estén cerca significa que dieron calificaciones similares a esas dos películas (preferencias parecidas en esas dimensiones).
- Que estén alejados significa que difieren en sus gustos respecto a esas películas.
- Limitación de usar solo dos películas: no captura la diversidad total de preferencias — la similitud observada puede ser engañosa si los usuarios solo coinciden en esas dos películas pero difieren en el resto.

## Actividad 4 — Distancia Minkowski y similitud coseno (ejemplo entre usuarios 1 y 5)

Resultados numéricos calculados sobre este dataset: 

- `mink_distance(user_dataset, 1, 5)` = 68.98550572402873 (p=2 → distancia euclidiana en el espacio completo de películas).
- `cos_similarity(user_dataset, 1, 5)` (usando todos los componentes, incluidos ceros) = 0.12907989044170284.

Interpretación:
- Una distancia pequeña indica que los vectores de calificaciones están próximos en el espacio (perfiles parecidos). Una distancia grande indica perfiles alejados.
- Una similitud coseno cercana a 1 indica que los vectores apuntan en direcciones muy parecidas (patrón de gustos similar, independientemente de la escala).
- Diferencia entre distancia y similitud: la distancia mide diferencia absoluta en el espacio (escala y dirección), la similitud coseno mide alineación de direcciones (independiente de magnitud).

## Actividad 5 — Corrección: comparar solo películas en común

- Por qué no conviene comparar películas que solo uno de los dos usuarios calificó: los ceros que representan 
 introducirían una penalización artificial si se usan directamente; esto castiga a usuarios por no haber visto una película en lugar de comparar sus verdaderas preferencias.
- Cómo cambia el resultado respecto a `cos_similarity`: concentrarse solo en `common_movies` (coordenadas donde ambos tienen rating>0) suele aumentar la similitud entre usuarios que comparten un subconjunto significativo de evaluaciones, porque elimina ruido introducido por ausencias. En el ejemplo, `cos_similarity` sobre todos los componentes fue ~0.129, mientras que la similitud corregida `similarity(user_dataset,1,5)` dio ~0.9708 (mucho mayor al comparar únicamente las películas ambas calificaron).
- ¿Qué representa `common_movies`?: una máscara booleana que indica las películas que ambos usuarios calificaron (rating > 0).
- ¿Qué pasa si no hay películas en común?: la función devuelve 0.0 (no es posible estimar similitud basada en intersección).

## Actividad 6 — Identificación de usuarios similares

Aplicando `top_users(user_dataset, 1, n=5)` sobre este conjunto de datos, los 5 usuarios más similares a `user 1` (según la similitud corregida) son:

- `[(1.0, 77), (1.0, 85), (1.0, 184), (1.0, 245), (1.0, 253)]`

- El usuario más similar a `user 1` es `user 77` con similitud `1.0` (hay varios usuarios emparejados con 1.0 en este dataset).

Comentarios sobre confiabilidad:
- Similitud exactamente 1.0 suele indicar perfiles idénticos en el subconjunto de películas en común (p. ej. mismos ratings en las mismas películas).
- Factores que afectan la confiabilidad: número de películas en común (pocos puntos en común producen estimaciones poco robustas), ruido en las calificaciones, popularidad de películas (items muy populares no discriminan), y sparsity del dataset.

## Actividad 7 — Recomendaciones desde el usuario más similar (user 77)

Películas que `user 1` no vio pero `user 77` calificó (ordenadas por la calificación del usuario similar). Las tres principales encontradas: 

- `movieId = 3996` → 
 (similar user's rating 5.0)
- `movieId = 4993` → 
 (5.0)
- `movieId = 5349` → 
 (5.0)

Por qué fueron seleccionadas: porque el usuario similar las calificó positivamente y el usuario objetivo no las había puntuado (candidatas naturales para recomendar).

Limitaciones de recomendar a partir de un solo usuario:
- Sesgo: las recomendaciones reflejan exclusivamente un solo perfil.
- Falta de robustez: si el vecino tiene gustos extremos o calificó pocas películas, las recomendaciones pueden ser inadecuadas.

Cómo mejorar: usar varios usuarios similares y combinar sus calificaciones (promedio ponderado por similitud), o aplicar filtrado híbrido que combine contenido y colaborativo.

## Actividad 8 — Promediando calificaciones de los 5 usuarios más similares

Al promediar las calificaciones de los 5 vecinos más similares (los mismos que aparecen en `top_users`), las cinco mejores películas candidatas fueron (IDs y títulos):

- `3996` → 
 (avg 5.0)
- `4993` → 
 (avg 5.0)
- `5349` → 
 (avg 5.0)
- `5378` → 
- `5952` → 
 (avg 5.0)

Observaciones:
- La lista es muy similar a la de un solo vecino en este caso concreto, porque los vecinos comparten calificaciones altas para los mismos títulos.
- Ventaja de usar varios usuarios: reduce el sesgo de un vecino y permite identificar películas que consistentemente son bien valoradas entre usuarios afines.
- Problema potencial: si los vecinos tienen pocas películas en común entre sí, el promedio estará basado en pocas observaciones y será menos fiable.
- Evaluación de razonabilidad: las películas listadas son populares y de alto perfil, por lo que parecen plausibles recomendaciones, pero conviene validar con mayor información (p. ej., historial más amplio del usuario).

## Actividad 9 — Exploración de géneros

- Los géneros en `movies_dataset['genres']` están separados por el carácter `|` (pipe).
- Sí, una película puede pertenecer a varios géneros (p. ej. `Adventure|Comedy`).
- Géneros presentes en el dataset (lista):

  Action, Adventure, Animation, Children, Comedy, Crime, Documentary, Drama, Fantasy, Film-Noir, Horror, IMAX, Musical, Mystery, Romance, Sci-Fi, Thriller, War, Western

- El género aporta información general sobre el contenido y ayuda a recomendar películas que comparten la misma categoría temática o estilo (útil para usuarios nuevos o cuando no hay históricos suficientes).

## Actividad 10 — Representación vectorial (one-hot)

- `genre_matrix`: cada fila representa una película; cada columna representa un género; un 1 indica presencia del género en esa película y un 0 indica ausencia.
- Esta representación permite comparar películas con operaciones vectoriales (producto punto, normas) y aplicar medidas como la similitud coseno.

## Actividad 11 — Similitud entre películas (coseno)

- Una similitud alta entre dos películas indica que comparten muchos géneros (sus vectores de género están alineados).
- Una similitud baja indica que tienen géneros distintos.
- La similitud coseno es adecuada porque mide alineación entre vectores binarios y es independiente de la magnitud (una película con muchos géneros no domina por tener mayor norma si ambas comparten las mismas dimensiones).
- Diferencia con comparar usuarios: al comparar usuarios se usan calificaciones (valores continuos con sparsity y magnitud variable); al comparar películas en este esquema las características son binarias y densidad mayor, y la interpretación es sobre contenido, no comportamiento.

## Actividad 12 — Películas similares a `Toy Story (1995)`

Top 5 películas más similares (por géneros), calculadas sobre la matriz de géneros del dataset: 

- `Antz (1998)` — similitud ≈ 1.0
- `Toy Story 2 (1999)` — similitud ≈ 1.0
- `Adventures of Rocky and Bullwinkle, The (2000)` — similitud ≈ 1.0
- `Emperor's New Groove, The (2000)` — similitud ≈ 1.0
- `Monsters, Inc. (2001)` — similitud ≈ 1.0

- ¿Comparten géneros?: sí — todas son películas de `Animation`, `Children` y con frecuencia `Comedy`, por eso la similitud de género es prácticamente 1.0.
- ¿Limitación?: usar solo géneros no captura subtemas, tono, director, actor o calidad; además, muchos filmes infantiles/animados comparten exactamente las mismas etiquetas de género, lo que lleva a similitud extrema incluso si la experiencia es diferente.

## Actividad 13 — Uso de etiquetas (tags)

- `tags_dataset` aporta información de texto libre: etiquetas descriptivas que pueden indicar tono (`funny`), tema (`time travel`), personajes (`superhero`), o valoraciones subjetivas (`must-see`).
- Diferencia con géneros: los géneros son categorías cerradas y estructuradas; las etiquetas son libres, más específicas y ricas, pero ruidosas.
- Problemas al trabajar con texto libre: sinónimos (`sci fi` vs `science fiction`), errores tipográficos, polisemia, etiquetado esporádico (sparsity) y ruido intencional.
- Cómo usarlas para mejorar recomendaciones: procesarlas con técnicas de NLP (normalización, lematización, TF–IDF, embeddings) para construir vectores semánticos por película; combinar estos vectores con las características de género y con señales colaborativas (híbrido).

## Conclusión

En esta práctica se implementaron dos enfoques complementarios: filtrado colaborativo (basado en similitud entre usuarios) y filtrado basado en contenido (basado en géneros). El filtrado colaborativo explota patrones en las calificaciones para recomendar películas que usuarios parecidos apreciaron; es potente pero depende de la densidad de calificaciones y sufre el problema del 
. El filtrado por contenido requiere un buen modelado de los atributos (géneros, etiquetas, sinopsis) y funciona bien para nuevos ítems pero carece de la perspectiva colectiva de preferencias.

Puntos clave: representar adecuadamente los datos (distinguiendo missing de valores reales), escoger medidas de similitud coherentes (coseno para vectores direccionales, distancia para magnitud), y lidiar con limitaciones prácticas (sparsity, ruido en etiquetas, necesidad de hibridación).

Si quieres, puedo: (1) incorporar estas respuestas directamente como celdas en `recommender_system_practice_notebook.ipynb`, (2) generar un PDF con el contenido de este notebook, o (3) crear pruebas automáticas que validen los valores numéricos calculados. ¿Cuál prefieres?